In [ ]:
import duckdb as db
import pandas as pd
import matplotlib.pyplot as plt

from scholar_rank.utils import PROJECT_ROOT
COMPACT_PATH = PROJECT_ROOT/'data'/'compact'
DB_PATH = "/data/math_english"

In [ ]:
con = db.connect(DB_PATH)

count = con.sql(f"""
    SELECT field, count(*) AS freq 
    FROM (
        SELECT *, topics[1].field_display_name AS field
        FROM works
    )
    GROUP BY field
""").fetchall()

con.close()

In [ ]:
con = db.connect()

count = con.sql(f"""
    SELECT language, count(*) AS freq
    FROM read_parquet('{COMPACT_PATH}/works/**/*.parquet') 
    GROUP BY language
    ORDER BY freq DESC
""").fetchall()

print(count)

con.close()

In [ ]:
# Getting maximum works id value
DB_PATH = '/data/math_english'

con = db.connect()

ids = con.sql(f"""
    SELECT regexp_replace(id, 'W', '')::BIGINT AS id,
    FROM read_parquet('{DB_PATH}/**/*.parquet') 
    ORDER BY id DESC
""")
print(DB_PATH)
print(ids.fetchone())
con.close()

In [ ]:
# Exploring tokenized data

CACHED_PATH = "/data/cached"

con = db.connect()

tokens = con.sql(f"""
    SELECT * FROM read_parquet('{CACHED_PATH}/**/*.parquet')
""")

print(tokens.columns)
print(f"Total entries: {con.sql("SELECT count(*) FROM tokens").fetchone()}")
print(f"Distinct documents: {con.sql("SELECT count(DISTINCT id) FROM tokens").fetchone()}")


token_count = con.sql(f"""
    SELECT token, count(token) as freq
    FROM tokens
    GROUP BY token
    ORDER BY freq DESC
""")

print(f"Distinct terms: {con.sql("SELECT count(DISTINCT token) FROM tokens").fetchone()}")

freq_table = token_count.fetchall()
for i in range(10):
    print(freq_table[i])